# Ternary JL certificate and parameter generation

This notebook has three executable cells. The first performs the security-parameter-independent computation. With $\mu_0=0.45$, a unit vector has at most four coordinates larger than $\mu_0$. After writing $x_j=w_j^2$, the remaining cases are finite-dimensional optimization problems over $\mu_0^2\leq x_j$ and $\sum_jx_j\leq1$. The code evaluates the exact finite sign sum for $\Phi_L$, bounds its variation using global gradient and Hessian estimates, and bisects boxes until every feasible box is below the envelope. All real-valued operations use `RealField(200)`; the branch-and-bound reserves a $2^{-100}$ safety margin. The resulting bound is $\Psi_{\rm exact}<0.550766101$, rounded upward to $\Psi=0.5508$ for the parameter formulas.

The second cell defines `ternary_jl_parameters(number_of_rows, security_parameter)`. For $N$ rows and target error $2^{-\lambda}$, it computes $\gamma_1$ by inverse `erfc`, $\alpha=(\log(2^{-(\lambda+1)})-N\log\Psi)/s_\times$, and $\beta$ by minimizing the Gaussian moment bound over integral moments. For the modular bound it chooses the largest admissible binomial threshold $T$, uses the per-row target $2^{-\lambda/N}$, balances modular Cases 2 and 3 in the auxiliary ratio, minimizes over $D$, and takes the next integer above the maximum of the three case bounds as $b$. The returned dictionary includes the intermediate constants and verification quantities, not only the values printed in the final table.

Using only $N=\lambda$ rows is not useful here: for the spike vector, $\Pr[Jw=0]=2^{-N}$ already consumes the entire failure budget, and the present transform bound cannot produce positive $\alpha$. In fact, positive $\alpha$ requires asymptotically $N/\lambda>\log 2/\log(1/\Psi)\approx1.1623$. We choose $N=2\lambda$ because it gives comfortable norm bounds and the simple modular per-row target $2^{-1/2}$. The helper nevertheless accepts every $N\geq\lambda$ and explicitly reports when the current analysis is infeasible. The third executable cell evaluates the five parameter sets used in the paper, applies conservative display rounding ($\gamma_1,\beta$ upward and $\alpha$ downward), and rechecks the modular inequalities with the displayed auxiliary constants.


In [1]:
from itertools import product

# Global constants and certificate: this cell is independent of lambda and the row count.
RR = RealField(200)
PI = RR.pi()

def bisect_root(f, lower, upper, iterations=240):
    lower, upper = RR(lower), RR(upper)
    f_lower, f_upper = f(lower), f(upper)
    if f_lower == 0:
        return lower
    if f_upper == 0:
        return upper
    if f_lower * f_upper > 0:
        raise ValueError("root is not bracketed")
    for _ in range(iterations):
        midpoint = (lower + upper) / 2
        f_midpoint = f(midpoint)
        if f_midpoint == 0:
            return midpoint
        if f_lower * f_midpoint > 0:
            lower, f_lower = midpoint, f_midpoint
        else:
            upper, f_upper = midpoint, f_midpoint
    return (lower + upper) / 2

def envelope(s):
    s = RR(s)
    return max((1 + exp(-s)) / 2, (1 + s)**(-RR(1) / 2))

S_CROSS = bisect_root(
    lambda s: (1 + s) * (1 + exp(-s))**2 - 4,
    RR("0.1"),
    RR("8"),
)
MU0 = RR("0.45")
MAX_DOMINANT = floor(1 / MU0**2)
CERTIFICATE_SAFETY = RR(2)**(-100)
KAPPA_BD = 1 / (2 * erfc(RR(1)))
KAPPA_BE = RR("0.52")

def signed_patterns(dimension):
    patterns = []
    for signs in product((-1, 0, 1), repeat=dimension):
        first_nonzero = next((sign for sign in signs if sign != 0), 0)
        if first_nonzero < 0:
            continue  # The sign-negated pattern has the same contribution.
        probability = prod(RR("0.5") if sign == 0 else RR("0.25") for sign in signs)
        if first_nonzero > 0:
            probability *= 2
        patterns.append((signs, probability))
    return patterns

def phi_and_gradient(coordinates, s, patterns):
    dimension = len(coordinates)
    rho_squared = max(RR(0), 1 - sum(coordinates))
    A = 1 + rho_squared * s
    roots = [sqrt(coordinate) for coordinate in coordinates]
    weighted_sum = RR(0)
    weighted_square_sum = RR(0)
    weighted_linear_sums = [RR(0)] * dimension
    for signs, probability in patterns:
        linear_form = sum(RR(signs[j]) * roots[j] for j in range(dimension))
        weight = probability * exp(-(s / A) * linear_form**2)
        weighted_sum += weight
        weighted_square_sum += weight * linear_form**2
        for j in range(dimension):
            weighted_linear_sums[j] += weight * linear_form * RR(signs[j])
    sqrt_A_inverse = A**(-RR(1) / 2)
    phi = sqrt_A_inverse * weighted_sum
    common = (s / 2) * A**(-RR(3) / 2) * weighted_sum
    common -= sqrt_A_inverse * (s**2 / A**2) * weighted_square_sum
    gradient = [
        sqrt_A_inverse * (-s / (roots[j] * A)) * weighted_linear_sums[j] + common
        for j in range(dimension)
    ]
    return phi, gradient

def certify_dominant_coordinates(dimension, mu0=MU0, s=S_CROSS, safety=CERTIFICATE_SAFETY):
    # In squared coordinates, the feasible set is a truncated simplex.
    # Each accepted box is controlled by both a Lipschitz bound and a
    # second-order Taylor bound; taking their minimum accelerates the search.
    patterns = signed_patterns(dimension)
    coordinate_floor = mu0**2
    target = envelope(s) - safety
    lipschitz = sqrt(s / PI) / (2 * mu0) + s / 2

    moment_1 = 2 * sqrt(s / PI)
    moment_2 = 2 * s
    moment_3 = 2 * sqrt(RR(2) / PI) * (2 * s)**(RR(3) / 2)
    moment_4 = 12 * s**2
    hessian_off_diagonal = (
        moment_4 / 16
        + moment_3 / (8 * mu0)
        + moment_2 / (16 * mu0**2)
    )
    hessian_diagonal = (
        moment_4 / 16
        + moment_3 / (8 * mu0)
        + moment_2 / (8 * mu0**2)
        + moment_1 / (8 * mu0**3)
    )
    hessian_row_bound = (dimension - 1) * hessian_off_diagonal + hessian_diagonal

    stack = [([coordinate_floor] * dimension, [RR(1)] * dimension)]
    certified_boxes = 0
    processed_boxes = 0
    largest_certified_upper_bound = RR(0)
    while stack:
        lower, upper = stack.pop()
        processed_boxes += 1
        if sum(lower) > 1:
            continue

        lower_sum = sum(lower)
        upper = [
            min(upper[j], 1 - (lower_sum - lower[j]))
            for j in range(dimension)
        ]
        midpoint = [(lower[j] + upper[j]) / 2 for j in range(dimension)]
        if sum(midpoint) <= 1:
            centre = midpoint
        else:
            # This is an L1-nearest feasible centre to the box midpoint.
            scale = (1 - lower_sum) / sum(midpoint[j] - lower[j] for j in range(dimension))
            centre = [lower[j] + scale * (midpoint[j] - lower[j]) for j in range(dimension)]
        radii = [
            max(centre[j] - lower[j], upper[j] - centre[j])
            for j in range(dimension)
        ]

        phi, gradient = phi_and_gradient(centre, s, patterns)
        radius_sum = sum(radii)
        linear_upper = phi + lipschitz * radius_sum
        quadratic_upper = phi + sum(abs(gradient[j]) * radii[j] for j in range(dimension))
        quadratic_upper += hessian_row_bound * radius_sum**2 / 2
        upper_bound = min(linear_upper, quadratic_upper)
        if upper_bound <= target:
            certified_boxes += 1
            largest_certified_upper_bound = max(largest_certified_upper_bound, upper_bound)
            continue

        widths = [upper[j] - lower[j] for j in range(dimension)]
        split_coordinate = max(range(dimension), key=lambda j: widths[j])
        if widths[split_coordinate] == 0:
            raise RuntimeError("certificate stalled at a zero-width box")
        split_point = (lower[split_coordinate] + upper[split_coordinate]) / 2
        lower_half_upper = list(upper)
        lower_half_upper[split_coordinate] = split_point
        upper_half_lower = list(lower)
        upper_half_lower[split_coordinate] = split_point
        stack.append((lower, lower_half_upper))
        stack.append((upper_half_lower, upper))

    return {
        "dimension": dimension,
        "certified": True,
        "certified_boxes": certified_boxes,
        "processed_boxes": processed_boxes,
        "largest_certified_upper_bound": largest_certified_upper_bound,
        "target": target,
    }

CERTIFICATES = []
for dimension in (2, 3, 4):
    print(f"Certifying {dimension} dominant coordinates ...")
    result = certify_dominant_coordinates(dimension)
    CERTIFICATES.append(result)
    print(f"  certified after processing {result['processed_boxes']} boxes")
ENVELOPE_AT_CROSSING = envelope(S_CROSS)
PEELING_ERROR = erfc(PI / (2 * MU0 * sqrt(S_CROSS)))
PSI_EXACT_BOUND = ENVELOPE_AT_CROSSING + PEELING_ERROR
PSI = RR("0.5508")  # The rounded-up value used by Section 4.
assert PSI_EXACT_BOUND < PSI

print(f"Real precision: {RR.precision()} bits")
print(f"s_cross = {S_CROSS}")
print(f"envelope(s_cross) = {ENVELOPE_AT_CROSSING}")
print(f"peeling error = {PEELING_ERROR}")
print(f"exact Psi bound = {PSI_EXACT_BOUND} < rounded Psi = {PSI}")
table(
    [[result["dimension"], result["certified_boxes"], result["processed_boxes"], result["largest_certified_upper_bound"]]
     for result in CERTIFICATES],
    header_row=["number of dominant coordinates", "certified boxes", "processed boxes", "largest accepted upper bound"],
)


Certifying 2 dominant coordinates ...
  certified after processing 645 boxes
Certifying 3 dominant coordinates ...


  certified after processing 9241 boxes
Certifying 4 dominant coordinates ...


  certified after processing 33687 boxes
Real precision: 200 bits
s_cross = 2.3105713567536942442022958568977868868312869999043273999942
envelope(s_cross) = 0.54960227708597349915109193565046647932506154764816296457832
peeling error = 0.0011638234379176986223688095702065416639400211936274524314987
exact Psi bound = 0.55076610052389119777346074522067302098900156884179041700982 < rounded Psi = 0.55080000000000000000000000000000000000000000000000000000000


number of dominant coordinates,certified boxes,processed boxes,largest accepted upper bound
\(2\),\(323\),\(645\),\(0.54951683596275646169169659682178443321319047904880128049972\)
\(3\),\(4621\),\(9241\),\(0.54960093794938899559583224354340193105571627687844246070335\)
\(4\),\(16844\),\(33687\),\(0.54959693223759947487383578354491116672974112934260951653682\)


In [2]:
def inverse_erfc(value):
    value = RR(value)
    if not (0 < value < 2):
        raise ValueError("inverse_erfc expects a value in (0, 2)")
    return bisect_root(lambda x: erfc(x) - value, RR(-64), RR(64))

def upper_moment_threshold(number_of_rows, failure_probability):
    # For N rows, the dominating Gaussian norm is Gamma(N/2, 1).
    # The candidates become increasing after the optimum; 32 subsequent
    # increases are used as a conservative termination check.
    shape = RR(number_of_rows) / 2
    failure_probability = RR(failure_probability)
    best_threshold = None
    best_moment = None
    consecutive_increases = 0
    hard_limit = 16 * (number_of_rows + ceil(-log(failure_probability) / log(RR(2)))) + 256
    for moment in range(1, hard_limit + 1):
        log_threshold = (
            log_gamma(shape + moment)
            - log_gamma(shape)
            - log(failure_probability)
        ) / moment
        threshold = exp(log_threshold)
        if best_threshold is None or threshold < best_threshold:
            best_threshold = threshold
            best_moment = moment
            consecutive_increases = 0
        elif moment > best_moment:
            consecutive_increases += 1
            if consecutive_increases >= 32:
                return best_threshold, best_moment
    raise RuntimeError("moment search reached its conservative hard limit")

def maximal_binomial_threshold(number_of_rows, security_parameter):
    # Return maximal T with Pr[Bin(N, 1/2) < T] <= 2^(-lambda).
    target = RR(2)**(-security_parameter)
    mass = RR(2)**(-number_of_rows)
    cumulative = RR(0)
    threshold = 0
    for index in range(number_of_rows + 1):
        next_cumulative = cumulative + mass
        if next_cumulative > target:
            break
        cumulative = next_cumulative
        threshold = index + 1
        if index < number_of_rows:
            mass *= RR(number_of_rows - index) / RR(index + 1)
    return threshold, cumulative

def modular_cases(D, ratio, ell, threshold_rows, per_row_target):
    A = ratio * D
    C = D / sqrt(1 - ratio**(-2))
    delta = KAPPA_BE * sqrt(RR(2)) / sqrt(ratio**2 - 1)
    tau = KAPPA_BD * erfc(D / 2)
    denominator = per_row_target - 2 * delta - tau
    if denominator <= 0:
        raise ValueError("the modular Case 3 denominator is not positive")
    case_2 = 2 * A * ell / sqrt(RR(threshold_rows))
    case_3 = 2 * C * ell / (sqrt(PI) * denominator)
    return {
        "A": A,
        "C": C,
        "delta": delta,
        "tau": tau,
        "case_2": case_2,
        "case_3": case_3,
    }

def balance_modular_cases(D, ell, threshold_rows, per_row_target):
    # For fixed D, Case 2 increases and Case 3 decreases with the ratio.
    # Their unique crossing therefore minimizes their maximum.
    tau = KAPPA_BD * erfc(D / 2)
    available_probability = per_row_target - tau
    if available_probability <= 0:
        raise ValueError("D is too small for the modular tail bound")
    ratio_floor = sqrt(1 + (2 * KAPPA_BE * sqrt(RR(2)) / available_probability)**2)
    lower = ratio_floor * (1 + RR(2)**(-100))

    def difference(ratio):
        cases = modular_cases(D, ratio, ell, threshold_rows, per_row_target)
        return cases["case_2"] - cases["case_3"]

    upper = max(RR(4), 2 * lower)
    while difference(upper) <= 0:
        upper *= 2
    ratio = bisect_root(difference, lower, upper, iterations=180)
    cases = modular_cases(D, ratio, ell, threshold_rows, per_row_target)
    return ratio, cases

def optimize_modular_constants(gamma, ell, threshold_rows, number_of_rows, security_parameter):
    # N=2*lambda specializes this target to 2^(-1/2), as in the paper.
    per_row_target = RR(2)**(-RR(security_parameter) / RR(number_of_rows))
    D_probability_floor = 2 * inverse_erfc(per_row_target / KAPPA_BD)
    D_floor = max(gamma, D_probability_floor)
    lower = D_floor + max(RR(1), abs(D_floor)) * RR(2)**(-100)
    upper = max(2 * gamma + 10, lower + 10)

    def objective(D, include_details=False):
        ratio, cases = balance_modular_cases(D, ell, threshold_rows, per_row_target)
        case_1 = ell / (1 - gamma / D)
        value = max(case_1, cases["case_2"], cases["case_3"])
        if include_details:
            return value, case_1, ratio, cases
        return value

    # The outer objective is the maximum of the three modular case bounds.
    # A high-precision golden-section search selects D.
    golden_ratio = (sqrt(RR(5)) - 1) / 2
    left, right = lower, upper
    x_left = right - golden_ratio * (right - left)
    x_right = left + golden_ratio * (right - left)
    f_left, f_right = objective(x_left), objective(x_right)
    for _ in range(180):
        if f_left <= f_right:
            right, x_right, f_right = x_right, x_left, f_left
            x_left = right - golden_ratio * (right - left)
            f_left = objective(x_left)
        else:
            left, x_left, f_left = x_left, x_right, f_right
            x_right = left + golden_ratio * (right - left)
            f_right = objective(x_right)
    D = (left + right) / 2
    minimum, case_1, ratio, cases = objective(D, include_details=True)
    strict_integer_bound = floor(minimum) + 1
    modular_row_probability = (
        2 * cases["C"] * ell / (RR(strict_integer_bound) * sqrt(PI))
        + 2 * cases["delta"]
        + cases["tau"]
    )
    assert RR(strict_integer_bound) > max(case_1, cases["case_2"], cases["case_3"])
    assert modular_row_probability < per_row_target
    return {
        "T": threshold_rows,
        "D": D,
        "ratio": ratio,
        "A": cases["A"],
        "C": cases["C"],
        "delta": cases["delta"],
        "tau": cases["tau"],
        "per_row_target": per_row_target,
        "case_1_bound": case_1,
        "case_2_bound": cases["case_2"],
        "case_3_bound": cases["case_3"],
        "b_minimum": minimum,
        "b": strict_integer_bound,
        "modular_row_probability": modular_row_probability,
    }

def ternary_jl_parameters(number_of_rows, security_parameter):
    # This is the public entry point for row counts other than 2*lambda.
    number_of_rows = Integer(number_of_rows)
    security_parameter = Integer(security_parameter)
    if security_parameter <= 0 or number_of_rows < security_parameter:
        raise ValueError("require number_of_rows >= security_parameter > 0")

    total_error = RR(2)**(-security_parameter)
    half_error = total_error / 2
    row_error = half_error / number_of_rows
    gamma = inverse_erfc(row_error / KAPPA_BD)
    alpha = (log(half_error) - RR(number_of_rows) * log(PSI)) / S_CROSS
    beta, upper_moment = upper_moment_threshold(number_of_rows, half_error)
    result = {
        "feasible": alpha > 0,
        "security_parameter": security_parameter,
        "number_of_rows": number_of_rows,
        "total_error": total_error,
        "half_error": half_error,
        "row_error": row_error,
        "mu0": MU0,
        "s_cross": S_CROSS,
        "envelope": ENVELOPE_AT_CROSSING,
        "peeling_error": PEELING_ERROR,
        "psi_exact_bound": PSI_EXACT_BOUND,
        "psi": PSI,
        "kappa_BD": KAPPA_BD,
        "kappa_BE": KAPPA_BE,
        "gamma1": gamma,
        "alpha": alpha,
        "beta": beta,
        "upper_moment": upper_moment,
    }
    if alpha <= 0:
        # The helper remains defined at N=lambda, but correctly returns no
        # lower norm factor or modular parameters when alpha is non-positive.
        result["reason"] = "the global transform bound gives no positive lower norm factor"
        result["ell"] = None
        result["u"] = sqrt(beta)
        result["modular"] = None
        return result

    ell = sqrt(alpha)
    threshold_rows, binomial_tail = maximal_binomial_threshold(number_of_rows, security_parameter)
    if threshold_rows == 0:
        result["feasible"] = False
        result["reason"] = "the binomial modular case has no admissible threshold"
        result["ell"] = ell
        result["u"] = sqrt(beta)
        result["modular"] = None
        return result

    modular = optimize_modular_constants(
        gamma, ell, threshold_rows, number_of_rows, security_parameter
    )
    modular["binomial_tail"] = binomial_tail
    result["ell"] = ell
    result["u"] = sqrt(beta)
    result["modular"] = modular
    return result


In [3]:
# These are the only concrete parameter sets used in the paper.
SECURITY_PARAMETERS = (64, 96, 128, 192, 256)
PARAMETER_SETS = [
    ternary_jl_parameters(2 * security_parameter, security_parameter)
    for security_parameter in SECURITY_PARAMETERS
]
assert all(parameters["feasible"] for parameters in PARAMETER_SETS)

# Directional rounding preserves the real tail inequalities. D and r are
# then chosen at two decimal places and checked again, since neither has a
# globally safe rounding direction.
PAPER_PARAMETERS = {
    64:  (RR("6.97"),  RR("13.53"), RR("171.8"), 16, RR("7.34"),  RR("5.37"), 73),
    96:  (RR("8.43"),  RR("20.45"), RR("257.5"), 23, RR("8.83"),  RR("5.99"), 100),
    128: (RR("9.66"),  RR("27.37"), RR("343.2"), 30, RR("10.08"), RR("6.53"), 126),
    192: (RR("11.75"), RR("41.21"), RR("514.6"), 44, RR("12.20"), RR("7.44"), 176),
    256: (RR("13.51"), RR("55.05"), RR("686.0"), 58, RR("13.98"), RR("8.22"), 224),
}

paper_rows = []
for parameters in PARAMETER_SETS:
    lam = parameters["security_parameter"]
    gamma, alpha, beta, threshold_rows, D, ratio, b = PAPER_PARAMETERS[lam]
    assert gamma >= parameters["gamma1"]
    assert alpha <= parameters["alpha"]
    assert beta >= parameters["beta"]
    assert threshold_rows == parameters["modular"]["T"]
    cases = modular_cases(
        D, ratio, sqrt(alpha), threshold_rows, parameters["modular"]["per_row_target"]
    )
    case_1 = sqrt(alpha) / (1 - gamma / D)
    assert RR(b) > max(case_1, cases["case_2"], cases["case_3"])
    paper_rows.append([
        lam, parameters["number_of_rows"], f"{gamma:.2f}", f"{alpha:.2f}",
        f"{beta:.1f}", threshold_rows, f"{D:.2f}", f"{ratio:.2f}", b,
    ])

table(
    paper_rows,
    header_row=["lambda", "rows", "gamma1", "alpha", "beta", "T", "D", "r", "b"],
)


lambda,rows,gamma1,alpha,beta,T,D,r,b
\(64\),\(128\),6.97,13.53,171.8,\(16\),7.34,5.37,\(73\)
\(96\),\(192\),8.43,20.45,257.5,\(23\),8.83,5.99,\(100\)
\(128\),\(256\),9.66,27.37,343.2,\(30\),10.08,6.53,\(126\)
\(192\),\(384\),11.75,41.21,514.6,\(44\),12.20,7.44,\(176\)
\(256\),\(512\),13.51,55.05,686.0,\(58\),13.98,8.22,\(224\)
